# Agent Platform 扩展演示 — 8 大组件能力端到端展示

4 个 Part，逐步展示 Google Agent Platform 生态系统的 **8 项核心能力**：

| Part | Steps | 展示能力 | 叙事 |
|:---:|:---:|---|---|
| **Part 1** | 1–5 | SPIFFE 身份 · Managed Runtime · IAM | Agent 有了身份，但没权限 |
| **Part 2** | 6–12 | Agent Registry · Auth Manager | Agent 要访问外部资源 |
| **Part 3** | 13–18 | 语义治理政策 | Agent 的行为合规吗？ |
| **Part 4** | 19–21 | 可观测性 · 生命周期管理 | 怎么证明这一切在工作？ |

---
# Part 1：核心 Demo — SPIFFE 身份 + IAM 政策

展示能力：SPIFFE 身份 · Managed Runtime · IAM

## Step 1：环境检查

验证 gcloud / ADC / Python。gcloud CLI 活跃账号和 ADC 可能不同。

In [ ]:
import shutil, subprocess, sys
print('Python :', sys.version.split()[0])
for tool in ('gcloud', 'gsutil'):
    print(f'{tool:7}: {shutil.which(tool) or "NOT FOUND"}')
def sh(cmd):
    try: return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except subprocess.CalledProcessError as e: return f'ERROR: {e.output.strip()}'
print('\ngcloud account :', sh('gcloud config get-value account'))
print('gcloud project :', sh('gcloud config get-value project'))
print('\nADC identity:')
print(sh('gcloud auth application-default print-access-token | head -c 1 > /dev/null && curl -s -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" https://openidconnect.googleapis.com/v1/userinfo'))

### Colab 认证

In [ ]:
from google.colab import auth
auth.authenticate_user()
import subprocess
subprocess.run(['gcloud', 'config', 'set', 'project', 'agent-identity-demo'], capture_output=True, text=True)
account = subprocess.run(['gcloud', 'config', 'get-value', 'account'], capture_output=True, text=True).stdout.strip()
project = subprocess.run(['gcloud', 'config', 'get-value', 'project'], capture_output=True, text=True).stdout.strip()
print(f'account: {account}\nproject: {project}')
tok = subprocess.run(['gcloud', 'auth', 'application-default', 'print-access-token'], capture_output=True, text=True)
print(f'ADC token: {"✅ OK" if tok.returncode == 0 else "❌ FAILED"}')

## Step 2：获取项目代码并配置

默认从 GitHub 克隆项目代码。

In [ ]:
import subprocess, os, shutil
REPO_DIR = '/content/agent-identity-extended'
GITHUB_REPO_URL = 'https://github.com/Leoric1/googleshare.git'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo already exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
else:
    if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
    print(f'Cloning from {GITHUB_REPO_URL}...')
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, REPO_DIR], capture_output=True, text=True)
    print('Cloned.')
os.chdir(REPO_DIR)
print(f'\n✅ cwd: {os.getcwd()}')
print(f'   文件列表:')
for f in sorted(os.listdir('.')):
    print(f'     {f}')

### 配置参数

编辑此 cell，填入你的项目信息。

In [ ]:
import os
PROJECT_ID = 'agent-identity-demo'
LOCATION = 'us-central1'
STAGING_BUCKET = ''
POLICY_BUCKET = ''
POLICY_PREFIX = 'policies/'
DISPLAY_NAME = 'policy-weather-agent'
AGENT_MODEL = 'gemini-2.5-flash'
OPENWEATHER_API_KEY = ''  # https://openweathermap.org/api
GEMINI_ENTERPRISE_APP_ID = ''
GEMINI_ENTERPRISE_LOCATION = 'global'
if not STAGING_BUCKET: STAGING_BUCKET = f'{PROJECT_ID}-agent-staging'
if not POLICY_BUCKET: POLICY_BUCKET = f'{PROJECT_ID}-policies'
assert PROJECT_ID != 'your-project-id', '❌ Set PROJECT_ID'
assert OPENWEATHER_API_KEY, '❌ Set OPENWEATHER_API_KEY'
for k in ('PROJECT_ID','LOCATION','STAGING_BUCKET','POLICY_BUCKET','DISPLAY_NAME','AGENT_MODEL'):
    print(f'  {k:30} = {eval(k)}')
print(f'  {"OPENWEATHER_API_KEY":30} = ***{OPENWEATHER_API_KEY[-4:]}')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION
os.environ['STAGING_BUCKET'] = STAGING_BUCKET
os.environ['POLICY_BUCKET'] = POLICY_BUCKET
os.environ['POLICY_PREFIX'] = POLICY_PREFIX
os.environ['DISPLAY_NAME'] = DISPLAY_NAME
os.environ['AGENT_MODEL'] = AGENT_MODEL
os.environ['EXTERNAL_API_URL'] = 'https://api.openweathermap.org/data/2.5/weather'
print('\n✅ Configuration set (Python vars + env vars)')

## Step 3：创建 GCS 存储桶并播种政策

Agent Identity principals 无法持有 Legacy Bucket 角色，必须用 `--uniform-bucket-level-access`。

In [ ]:
%cd /content/agent-identity-extended
!gcloud services enable aiplatform.googleapis.com storage.googleapis.com discoveryengine.googleapis.com
!gcloud storage buckets create gs://{STAGING_BUCKET} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage buckets create gs://{POLICY_BUCKET} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp sample_policies/*.md gs://{POLICY_BUCKET}/policies/

## Step 4：部署到 Agent Engine — 铸造 SPIFFE 身份

`identity_type=AGENT_IDENTITY` → 部署即铸造，删除即失效

In [ ]:
%cd /content/agent-identity-extended
!pip install -e . -q
!python deploy.py 2>&1 | tee /tmp/deploy_output.txt

### 获取 SPIFFE Principal

```
principal://agents.global.org-{ORG_ID}.system.id.goog/resources/aiplatform/projects/{PROJECT}/locations/{LOC}/reasoningEngines/{ENGINE_ID}
             └── TRUST DOMAIN ──┘                                  └── SERVICE ──┘  └── RESOURCE PATH ──┘
```

In [ ]:
import subprocess, json, re, os
with open('/tmp/deploy_output.txt') as f: deploy_output = f.read()
match = re.search(r'reasoningEngine:\s*(.+)', deploy_output)
REASONING_ENGINE = match.group(1).strip() if match else ''
print(f'Reasoning Engine: {REASONING_ENGINE}')
os.environ['REASONING_ENGINE'] = REASONING_ENGINE
token = subprocess.check_output('gcloud auth application-default print-access-token', shell=True, text=True).strip()
resp = subprocess.check_output(f'curl -s -H "Authorization: Bearer {token}" https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}', shell=True, text=True)
engine = json.loads(resp)
effective_identity = engine['spec']['effectiveIdentity']
AGENT_PRINCIPAL = f'principal://{effective_identity}'
print(f'\n✅ Agent SPIFFE Principal:\n   {AGENT_PRINCIPAL}')
parts = effective_identity.split('/')
print(f'\n拆解:')
print(f'   Trust Domain  : {parts[0]}')
print(f'   Service        : {parts[2]}')
print(f'   Resource Path  : /{"/".join(parts[2:])}')

## Step 5：IAM 政策 — 先失败，再修复

1. **测试** → 403 失败（有身份无权限）
2. **绑定 IAM** → `roles/storage.objectViewer` 到 SPIFFE principal
3. **重新测试** → 成功

### 5a：测试 — 预期失败（403）

In [ ]:
%cd /content/agent-identity-extended
!python remote_test.py "What is our remote work policy on equipment stipend?" 2>&1 | head -50

### 5b：绑定 IAM 角色

In [ ]:
!gcloud storage buckets add-iam-policy-binding gs://{POLICY_BUCKET} \
  --member="{AGENT_PRINCIPAL}" \
  --role="roles/storage.objectViewer" \
  --project={PROJECT_ID}
print('\n✅ IAM binding applied — allow ~300 seconds for propagation')

### 5c：重新测试 — 应该成功

In [ ]:
import time
print('等待 IAM 传播（10秒）...')
time.sleep(10)
%cd /content/agent-identity-extended
!python remote_test.py "What is our remote work policy on equipment stipend?" 2>&1 | head -50

---
# Part 2：Agent Registry + Auth Manager

Agent Registry 管"有哪些资源"，Auth Manager 管"怎么访问"。

## Step 6：创建 Auth Provider — Auth Manager

API Key 存在 Auth Manager 保险库中，不在 Agent 代码里。ADK 运行时透明注入。

In [ ]:
AUTH_PROVIDER_NAME = 'openweather-apikey'
AUTH_PROVIDER_RESOURCE = f'projects/{PROJECT_ID}/locations/{LOCATION}/authProviders/{AUTH_PROVIDER_NAME}'
!gcloud agent-identity auth-providers create {AUTH_PROVIDER_NAME} \
  --location={LOCATION} --api-key="{OPENWEATHER_API_KEY}" \
  --description="OpenWeatherMap API Key" --project={PROJECT_ID}
os.environ['AUTH_PROVIDER_NAME'] = AUTH_PROVIDER_RESOURCE
print(f'\n✅ Auth Provider: {AUTH_PROVIDER_RESOURCE}')
print(f'   API Key: ***{OPENWEATHER_API_KEY[-4:]}')

## Step 7：注册 Endpoint — Agent Registry

注册后 Gateway 拦截请求时能看到 Endpoint 元数据，可做工具级控制。

In [ ]:
ENDPOINT_NAME = 'openweather-endpoint'
!gcloud agent-registry services create {ENDPOINT_NAME} \
  --project={PROJECT_ID} --location={LOCATION} \
  --display-name="OpenWeatherMap API" \
  --endpoint-spec-type=no-spec \
  --interfaces=url=https://api.openweathermap.org/data/2.5/weather,protocolBinding=http-json
print(f'\n✅ Endpoint registered: {ENDPOINT_NAME}')

## Step 8：搜索已注册的 Agent — Agent Registry

Agent 在部署到 Agent Engine 时已自动注册到 Agent Registry。这里搜索并展示已注册的 Agent。

In [ ]:
AGENT_REGISTRY_NAME = 'policy-weather-agent'
print('=== 搜索已注册的 Agent ===')
!gcloud agent-registry agents search \
  --project={PROJECT_ID} --location={LOCATION} \
  --search-string="policy OR weather"
print('\n=== 列出所有已注册的 Agent ===')
!gcloud agent-registry agents list \
  --project={PROJECT_ID} --location={LOCATION}

## Step 9：创建 Agent ↔ Auth Provider 绑定

绑定后 ADK 自动解析凭据，无需代码中手动定义。

In [ ]:
BINDING_NAME = 'policy-weather-openweather-binding'
!gcloud agent-registry bindings create {BINDING_NAME} \
  --source-identifier="urn:agent:avnit:demo:policy-weather-agent" \
  --target-identifier="urn:endpoint:avnit:demo:openweather-endpoint" \
  --auth-provider-binding="{AUTH_PROVIDER_RESOURCE}" \
  --project={PROJECT_ID} --location={LOCATION}
print(f'\n✅ Binding: policy-weather-agent → {AUTH_PROVIDER_RESOURCE}')

## Step 10：搜索 Registry 中的资源

In [ ]:
print('=== 搜索 Agent ===')
!gcloud agent-registry agents search --project={PROJECT_ID} --location={LOCATION} --search-string="policy OR weather"
print('\n=== 列出 Endpoint ===')
!gcloud agent-registry services list --project={PROJECT_ID} --location={LOCATION}
print('\n=== 列出 Binding ===')
!gcloud agent-registry bindings list --project={PROJECT_ID} --location={LOCATION}

## Step 11：代码对比 — GCP 透明拦截 vs 传统方式

### GCP 透明拦截（本 demo）
```python
# 函数内零行认证代码
def get_weather(city):
    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&units=metric'
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

# ADK 自动注入凭据
get_weather_tool = AuthenticatedFunctionTool(func=get_weather, auth_config=auth_config)
```

### 传统方式（AWS 风格）
```python
@requires_access_token(provider='my-provider')
async def get_weather(lat, lon):
    token = get_token_from_context()               # ← 取令牌
    headers = {'X-Api-Key': token}                 # ← 拼 Header
    response = await client.get(url, headers=headers)  # ← 传 Header
    return response.json()
```

| 维度 | GCP 透明拦截 | 传统装饰器 |
|---|---|---|
| 函数内认证代码 | **0 行** | 3 行 |
| 开发者接触令牌 | **否** | 是 |
| 令牌存储 | Auth Manager 保险库 | Agent 进程内存 |

## Step 12：重新部署 + 测试外部 API

重新部署（带 Auth Provider），测试策略 + 天气两个场景。

In [ ]:
import os
os.environ['AUTH_PROVIDER_NAME'] = AUTH_PROVIDER_RESOURCE
os.environ['OPENWEATHER_API_KEY'] = OPENWEATHER_API_KEY
%cd /content/agent-identity-extended
!python deploy.py 2>&1 | tee /tmp/deploy_output2.txt

In [ ]:
import subprocess, json, re, os
with open('/tmp/deploy_output2.txt') as f: txt = f.read()
m = re.search(r'reasoningEngine:\s*(.+)', txt)
REASONING_ENGINE = m.group(1).strip() if m else REASONING_ENGINE
os.environ['REASONING_ENGINE'] = REASONING_ENGINE
print(f'New Engine: {REASONING_ENGINE}')
token = subprocess.check_output('gcloud auth application-default print-access-token', shell=True, text=True).strip()
resp = subprocess.check_output(f'curl -s -H "Authorization: Bearer {token}" https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}', shell=True, text=True)
engine = json.loads(resp)
AGENT_PRINCIPAL = f'principal://{engine["spec"]["effectiveIdentity"]}'
print(f'New Principal: {AGENT_PRINCIPAL}')
!gcloud storage buckets add-iam-policy-binding gs://{POLICY_BUCKET} \
  --member="{AGENT_PRINCIPAL}" --role="roles/storage.objectViewer" --project={PROJECT_ID}
print('✅ IAM binding applied')

In [ ]:
import time
print('等待 IAM 传播（10秒）...')
time.sleep(10)
print('=== 测试 1：策略问题 ===')
!python remote_test.py "What is our data retention policy for email?" 2>&1 | head -30
print('\n=== 测试 2：天气问题（Auth Manager 透明注入）===')
!python remote_test.py "What is the current weather in London?" 2>&1 | head -30

---
# Part 3：语义治理政策

## IAM vs 语义治理
| 维度 | IAM | 语义治理 |
|---|---|---|
| 解决的问题 | 有没有权限 | 应不应该做 |
| 评估方式 | CEL 确定性评估 | LLM 语义评估 |
| 规则语言 | 编程式 | 自然语言 (NLC) |
| 执行时机 | 请求到达时 | LLM 返回工具调用建议后、执行前 |
| 检查内容 | 身份+权限 | ① 用户意图一致性 ② 组织合规性 |

## Step 13：[Console] 创建 Agent Gateway

Agent Gateway 必须是 **Google-managed** 的，需要通过 Console 创建。

### 操作步骤

1. 打开 [Google Cloud Console](https://console.cloud.google.com)
2. 导航到 **Agent Platform → 治理（Govern）→ Agent Gateway**
3. 点击 **创建**
4. 填入以下配置：

| 字段 | 值 |
|---|---|
| 名称 | `policy-weather-gateway` |
| 区域 | `us-central1` |
| 部署模式 | 由 Google 管理（默认，不可更改） |
| 智能体注册库 | 区域注册表 |
| 受控访问路径 | 代理到任意目的地（出站流量） |
| 访问授权 | 强制执行政策 |
| 政策模型 | 统一访问权限政策 |
| 启用 Model Armor | 不打开 |

5. 点击 **创建**，等待 Gateway 状态变为 **ACTIVE**

> **注意**：Agent Gateway 是 Google-managed 的基础设施，`gcloud` CLI 不支持 `create` 命令，
> 必须通过 Console 创建。通过 REST API 创建的普通 Gateway 不被 Agent Engine 接受
> （会报 "must be Google-managed" 错误）。

创建完成后，运行下面的 cell 确认 Gateway 名称：

In [ ]:
GATEWAY_NAME = 'policy-weather-gateway'
print(f'✅ Gateway name set: {GATEWAY_NAME}')
print('   请确认已在 Console 中创建此 Gateway')

## Step 14：绑定 Agent 到 Gateway（出站流量）

In [ ]:
import subprocess, json, urllib.request, urllib.error
token = subprocess.check_output('gcloud auth print-access-token', shell=True, text=True).strip()
url = f'https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}?updateMask=spec.deploymentSpec.agentGatewayConfig'
payload = {'spec': {'deploymentSpec': {'agentGatewayConfig': {'agentToAnywhereConfig': {'agentGateway': f'projects/{PROJECT_ID}/locations/{LOCATION}/agentGateways/{GATEWAY_NAME}'}}}}}}
body = json.dumps(payload).encode('utf-8')
req = urllib.request.Request(url, data=body, method='PATCH', headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'})
try:
    with urllib.request.urlopen(req) as resp:
        print(f'HTTP {resp.status}\n{resp.read().decode()[:500]}')
        print('\n✅ Agent bound to Gateway')
except urllib.error.HTTPError as e:
    print(f'HTTP {e.code}\n{e.read().decode()[:500]}')
    print('\n⚠️ Gateway binding failed — check if API is enabled')

## Step 15：预配 Policy Engine + 配置 NLC 规则

### 15a：预配 Policy Engine（Colab）

使用 gcloud 预配语义治理政策引擎。

In [ ]:
!gcloud services enable \
    aiplatform.googleapis.com \
    agentregistry.googleapis.com \
    networkservices.googleapis.com \
    networksecurity.googleapis.com \
    compute.googleapis.com \
    dns.googleapis.com \
    --project={PROJECT_ID}
!gcloud beta ai semantic-governance-policy-engine update \
    --location={LOCATION} \
    --project={PROJECT_ID}
!gcloud beta ai semantic-governance-policy-engine describe \
    --location={LOCATION} \
    --project={PROJECT_ID}

### 15b：[Console] 配置 NLC 规则

1. 打开 Console → Agent Platform → 治理 → 语义治理政策
2. 点击 **创建政策**

| 字段 | 值 |
|---|---|
| 政策 ID | `policy-only-weather-restriction` |
| 说明 | 限制 Agent 仅回答公司政策和天气问题 |
| 代理选择 | `policy-weather-agent` |

### NLC 限制条件（自然语言）
```
This agent is only authorized to answer questions about:
1. Internal company policies (acceptable use, data retention, remote work)
2. Current weather conditions for any city

The agent MUST NOT:
- Execute tool calls for questions unrelated to these two topics
- Provide financial advice, medical recommendations, or legal counsel
- Access or modify data outside the policy bucket
- Make purchases or transactions
```

### NLC 工作原理
```
用户提问 → LLM 返回工具调用建议 → Gateway 拦截
     ↓                                        ↓
  Policy Engine 评估                    发送 NLC + 聊天记录
  ① 用户意图一致性检查
  ② 组织限制条件检查
     ↓
  ALLOW → 执行工具调用
  DENY  → 移除工具调用，返回拒绝原因
```

## Step 16：测试 — 违反 NLC → 预期 DENY

In [ ]:
print('=== 测试：违反 NLC（预期 DENY）===')
print('问题: 帮我买一张从北京到上海的机票\n')
%cd /content/agent-identity-extended
!python remote_test.py "帮我买一张从北京到上海的机票" 2>&1 | head -50

## Step 17：测试 — 合规问题 → 预期 ALLOW

In [ ]:
print('=== 测试：合规问题（预期 ALLOW）===')
print('问题: What is our acceptable use policy on password sharing?\n')
%cd /content/agent-identity-extended
!python remote_test.py "What is our acceptable use policy on password sharing?" 2>&1 | head -50

---
# Part 4：可观测性 + 清理

## Step 18：审计日志 — 按 SPIFFE Principal 过滤

证明确实是 Agent Identity（非服务账号/人类）在访问 GCS。

In [ ]:
import json, subprocess, tempfile, os
policy_json = subprocess.check_output(f'gcloud projects get-iam-policy {PROJECT_ID} --format=json', shell=True, text=True)
policy = json.loads(policy_json)
desired = {'storage.googleapis.com': ['ADMIN_READ','DATA_READ','DATA_WRITE'], 'aiplatform.googleapis.com': ['ADMIN_READ','DATA_READ','DATA_WRITE']}
existing = {a['service']: a for a in policy.get('auditConfigs', [])}
for service, lts in desired.items():
    cfg = existing.get(service, {'service': service, 'auditLogConfigs': []})
    have = {c['logType'] for c in cfg['auditLogConfigs']}
    for lt in lts:
        if lt not in have: cfg['auditLogConfigs'].append({'logType': lt})
    existing[service] = cfg
policy['auditConfigs'] = list(existing.values())
with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as f:
    json.dump(policy, f); policy_path = f.name
subprocess.run(f'gcloud projects set-iam-policy {PROJECT_ID} {policy_path} --quiet', shell=True, capture_output=True, text=True)
os.unlink(policy_path)
print('✅ Data Access audit logs enabled')

### 三条关键日志查询（在 Logs Explorer 中执行）

**1. Agent stdout / 工具错误**
```
resource.type="aiplatform.googleapis.com/ReasoningEngine" AND resource.labels.reasoning_engine_id="{ENGINE_ID}"
```

**2. IAM 拒绝审计（Step 5a 的 403）**
```
protoPayload.serviceName="storage.googleapis.com" AND protoPayload.resourceName:"buckets/{POLICY_BUCKET}" AND severity=ERROR
```

**3. Agent Principal 读取 GCS（证明 Agent Identity 在工作）**
```
protoPayload.serviceName="storage.googleapis.com" AND protoPayload.authenticationInfo.principalSubject="spiffe://agents.global.org-..." AND protoPayload.resourceName:"buckets/{POLICY_BUCKET}"
```

## Step 19：Agent Gateway 日志 — 语义治理评估记录

In [ ]:
import json
print('=== Gateway 日志查询 ===')
print(f'resource.type="networkservices.googleapis.com/Gateway"\nresource.labels.location="{LOCATION}"\nresource.labels.gateway_name="{GATEWAY_NAME}"')
print('\n关键字段:')
print(json.dumps({'severity':'INFO/ERROR','httpRequest':'HTTP请求','jsonPayload.agentGatewayInfo.mcpInfo':'MCP方法/工具','jsonPayload.agentGatewayInfo.agentRegistryResource':'Registry资源名','jsonPayload.serviceExtensionsInfo':'授权扩展信息'}, indent=2, ensure_ascii=False))

## Step 20：清理资源

按依赖顺序删除所有创建的资源，停止计费。

删除引擎 → SPIFFE 身份自动失效。

In [ ]:
import subprocess, json, urllib.request, urllib.error, time
token = subprocess.run(['gcloud', 'auth', 'application-default', 'print-access-token'],
                       capture_output=True, text=True).stdout.strip()

# 1. 删除所有 Reasoning Engines
print('=== 删除 Reasoning Engines ===')
r = subprocess.run(['curl', '-s', '-H', f'Authorization: Bearer {token}',
    f'https://{LOCATION}-aiplatform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines'],
    capture_output=True, text=True)
data = json.loads(r.stdout)
for engine in data.get('reasoningEngines', []):
    name = engine['name']
    subprocess.run(['curl', '-s', '-X', 'DELETE', '-H', f'Authorization: Bearer {token}',
        f'https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{name}?force=true'],
        capture_output=True, text=True)
    print(f'  Deleted engine: {name.split("/")[-1]}')

# 2. 删除授权政策
print('\n=== 删除授权政策 ===')
r2 = subprocess.run(['gcloud', 'network-security', 'authz-policies', 'list',
    '--project', PROJECT_ID, '--location', LOCATION],
    capture_output=True, text=True)
if r2.stdout.strip() and 'Listed 0' not in r2.stdout:
    for line in r2.stdout.strip().split('\n'):
        if line and not line.startswith('NAME') and not line.startswith('Listed'):
            policy_name = line.split()[0]
            subprocess.run(['gcloud', 'network-security', 'authz-policies', 'delete', policy_name,
                '--project', PROJECT_ID, '--location', LOCATION, '--quiet'],
                capture_output=True, text=True)
            print(f'  Deleted policy: {policy_name}')
else:
    print('  No authz policies found')

# 3. 删除 Agent Gateways
print('\n=== 删除 Agent Gateways ===')
for gw_name in [GATEWAY_NAME, 'policy-weather-gateway-2']:
    subprocess.run(['gcloud', 'network-services', 'agent-gateways', 'delete', gw_name,
        '--project', PROJECT_ID, '--location', LOCATION, '--quiet'],
        capture_output=True, text=True)
    print(f'  Deleted gateway: {gw_name}')

# 4. 删除 Registry 资源
print('\n=== 删除 Registry 资源 ===')
subprocess.run(['gcloud', 'agent-registry', 'bindings', 'delete', BINDING_NAME,
    '--project', PROJECT_ID, '--location', LOCATION, '--quiet'],
    capture_output=True, text=True)
print(f'  Deleted binding: {BINDING_NAME}')
subprocess.run(['gcloud', 'agent-registry', 'services', 'delete', ENDPOINT_NAME,
    '--project', PROJECT_ID, '--location', LOCATION, '--quiet'],
    capture_output=True, text=True)
print(f'  Deleted endpoint: {ENDPOINT_NAME}')

# 5. 删除 Auth Provider
print('\n=== 删除 Auth Provider ===')
subprocess.run(['gcloud', 'agent-identity', 'auth-providers', 'delete', AUTH_PROVIDER_NAME,
    '--project', PROJECT_ID, '--location', LOCATION, '--quiet'],
    capture_output=True, text=True)
print(f'  Deleted auth provider: {AUTH_PROVIDER_NAME}')

# 6. 删除 GCS 桶
print('\n=== 删除 GCS 桶 ===')
subprocess.run(['gcloud', 'storage', 'rm', '-r', f'gs://{STAGING_BUCKET}'],
    capture_output=True, text=True)
print(f'  Deleted bucket: {STAGING_BUCKET}')
subprocess.run(['gcloud', 'storage', 'rm', '-r', f'gs://{POLICY_BUCKET}'],
    capture_output=True, text=True)
print(f'  Deleted bucket: {POLICY_BUCKET}')

# 7. 取消预配 Policy Engine
print('\n=== 取消预配 Policy Engine ===')
subprocess.run(['gcloud', 'beta', 'ai', 'semantic-governance-policy-engine', 'deprovision',
    '--location', LOCATION, '--project', PROJECT_ID, '--quiet'],
    capture_output=True, text=True)
print('  Deprovisioned policy engine')

print('\n✅ 所有资源清理完成')
print('✅ Agent SPIFFE 身份随引擎删除自动失效')
print('\n⚠️ 请在 Console 中手动删除语义治理政策: policy-only-weather-restriction')

---
## Troubleshooting

| Error | Fix |
|---|---|
| `403 storage.buckets.get` during deploy | ADC ≠ gcloud identity. `gcloud auth application-default login --account=<right>` |
| `403 storage.objects.list denied` | IAM binding not applied. Re-check principal |
| `gcloud agent-registry` not found | `gcloud services enable agentregistry.googleapis.com` |
| `gcloud agent-identity` not found | `gcloud services enable agentidentity.googleapis.com` |
| Weather API 401 | Auth Provider not created. `gcloud agent-identity auth-providers describe` |
| Semantic DENY not working | Check: Gateway bound, Policy Engine active, NLC rule configured |
| `AuthenticatedFunctionTool` import error | Ensure `google-cloud-aiplatform[adk,agent_engines]` in requirements |
| Gateway binding `must be Google-managed` | 必须通过 Console 创建 Agent Gateway，不能用 REST API 创建 |